# Train MobileDeepFilterNet (ready-to-run)

### Rationale (mixing strategy)

- **Triangular SNR** biases sampling toward challenging −10→0 dB scenes often seen in real environments.
- **Multi-noise** (1–3) and **time-varying SNR** model non-stationary backgrounds and sudden transitions.
- **RIR on clean** approximates room reverberation in captured speech.
- **Per-subsegment SNR** ensures robustness to changing acoustic conditions.


## Install dependencies

In [ ]:
!pip install torch torchaudio numpy scipy librosa soundfile matplotlib onnx onnxruntime tensorboard pystoi pyroomacoustics kagglehub pyyaml

# Optional PESQ: may fail depending on OS/toolchain.
try:
    import pesq  # noqa: F401
except Exception as e:
    print("[note] PESQ not available; will skip PESQ metric. Error:", e)
    print("       You can try: pip install pesq")

## Import project + show config

In [ ]:
import sys
from pathlib import Path

ROOT = Path(r"E:\0. Noise reduce")
sys.path.insert(0, str(ROOT))

import yaml

cfg_path = ROOT / "configs" / "train.yaml"
cfg = yaml.safe_load(cfg_path.read_text(encoding="utf-8"))
cfg

## Verify data paths / download script

Defaults:
- Clean: `PROJECT_ROOT/data/clean`
- Noise (explicit): `E:\0. Noise reduce\denoise-audio\data\processed\noise_realtime`


In [ ]:
clean_root = Path(cfg.get("clean_root", ROOT / "data" / "clean"))
noise_root = Path(cfg.get("noise_root", Path(r"E:\0. Noise reduce\denoise-audio\data\processed\noise_realtime")))

print("Clean root:", clean_root)
print("Noise root:", noise_root)

if not clean_root.exists() or not noise_root.exists():
    print("\n[error] Missing datasets.")
    if not clean_root.exists():
        print("- Clean dataset missing at:", clean_root)
    if not noise_root.exists():
        print("- Noise dataset missing at:", noise_root)
    print("\nRun this KaggleHub downloader:")
    print(f"python \"{ROOT / 'scripts' / 'download_dataset.py'}\"")
    print("\nClean only:")
    print(f"python \"{ROOT / 'scripts' / 'download_dataset.py'}\" --clean")
    print("Noise only:")
    print(f"python \"{ROOT / 'scripts' / 'download_dataset.py'}\" --noise")
    print("Skip MUSAN:")
    print(f"python \"{ROOT / 'scripts' / 'download_dataset.py'}\" --no-musan")
else:
    print("\n[ok] Data paths exist.")

## Build dataloaders (train: on-the-fly mixing, val: deterministic manifest)

In [ ]:
from src.utils import list_audio_files, generate_val_manifest
from src.dataset import NoiseSuppressionDataset
from torch.utils.data import DataLoader
import torch

clean_list = list_audio_files(clean_root)
noise_list = list_audio_files(noise_root)
print("#clean:", len(clean_list))
print("#noise:", len(noise_list))

assert len(clean_list) > 0 and len(noise_list) > 0

manifest_path = ROOT / "manifests" / "val_manifest.jsonl"
if not manifest_path.exists():
    generate_val_manifest(clean_list, noise_list, manifest_path, n_val=int(cfg.get("n_val", 500)), seed=int(cfg.get("seed", 42)), segment_len=float(cfg.get("segment_len", 4.0)))

mix_cfg = {
    "seed": int(cfg.get("seed", 42)),
    "p_rir": float(cfg.get("p_rir", 0.35)),
    "p_vary": float(cfg.get("p_vary", 0.4)),
    "project_root": str(ROOT),
}

train_ds = NoiseSuppressionDataset(clean_list, noise_list, segment_len=float(cfg["segment_len"]), sr=int(cfg["sr"]), mix_config=mix_cfg, mode="train")
val_ds = NoiseSuppressionDataset(clean_list, noise_list, segment_len=float(cfg["segment_len"]), sr=int(cfg["sr"]), mix_config=mix_cfg, mode="val", manifest=manifest_path)

# DEBUG: Check sample shapes from dataset
print("\n[DEBUG] Inspecting dataset samples...")
for i in range(min(5, len(train_ds))):
    sample = train_ds[i]
    print(f"Sample {i}: clean={sample['clean'].shape}, noisy={sample['noisy'].shape}")

# Custom collate function to handle variable-length sequences
def collate_audio_batch(batch):
    """Pad variable-length audio samples to the same length"""
    # Get max length in this batch
    max_clean_len = max(item['clean'].shape[-1] for item in batch)
    max_noisy_len = max(item['noisy'].shape[-1] for item in batch)
    max_len = max(max_clean_len, max_noisy_len)
    
    # Pad all to max length
    batch_clean = []
    batch_noisy = []
    batch_meta = []
    
    for item in batch:
        clean = item['clean']
        noisy = item['noisy']
        
        # Pad if needed
        if clean.shape[-1] < max_len:
            pad_len = max_len - clean.shape[-1]
            clean = torch.nn.functional.pad(clean, (0, pad_len))
        if noisy.shape[-1] < max_len:
            pad_len = max_len - noisy.shape[-1]
            noisy = torch.nn.functional.pad(noisy, (0, pad_len))
        
        batch_clean.append(clean)
        batch_noisy.append(noisy)
        batch_meta.append(item.get('meta', ''))
    
    return {
        'clean': torch.stack(batch_clean),
        'noisy': torch.stack(batch_noisy),
        'meta': batch_meta
    }

print("\n[INFO] Using custom collate function to handle variable-length audio")
print("[WARNING] Using num_workers=0 (single process) to avoid multiprocessing issues on Windows")

# Use num_workers=0 to avoid multiprocessing issues on Windows
train_loader = DataLoader(train_ds, batch_size=int(cfg["batch_size"]), shuffle=True, num_workers=0, pin_memory=False, drop_last=True, collate_fn=collate_audio_batch)
val_loader = DataLoader(val_ds, batch_size=int(cfg["batch_size"]), shuffle=False, num_workers=0, pin_memory=False, collate_fn=collate_audio_batch)

batch = next(iter(train_loader))
print(f"\n[SUCCESS] Batch loaded:")
print(f"  clean: {batch['clean'].shape}")
print(f"  noisy: {batch['noisy'].shape}")
print(f"  meta: {batch['meta'][0] if isinstance(batch['meta'], list) else '(meta batched)'}")


## K-Fold Cross-Validation Setup (train/val/test split)

This implements k-fold stratification where each fold has:
- **Test**: 1 fold (held out entirely)
- **Train**: (k-2) folds (on-the-fly mixing)
- **Val**: 1 fold (deterministic validation)



## Generate All Fold Manifests (prepare data for all folds)

Pre-generate validation and test manifests for all k-fold splits before training.


In [ ]:
# Load fold indices (generated from k-fold setup cell)
import json

fold_indices_path = ROOT / "manifests" / "fold_indices.json"
with open(fold_indices_path, "r") as f:
    fold_indices = json.load(f)

n_splits = len(fold_indices)
fold_dir = ROOT / "manifests"

print(f"Pre-generating manifests for all {n_splits} folds...")

for fold_idx in range(n_splits):
    fold_info = fold_indices[fold_idx]
    train_val_idx = fold_info["train_val"]
    test_idx = fold_info["test"]
    
    # Split train_val into train and val
    val_size = max(1, len(train_val_idx) // 5)
    train_idx = train_val_idx[:-val_size]
    val_idx = train_val_idx[-val_size:]
    
    # Get clean files for this fold
    val_clean = [clean_list[i] for i in val_idx]
    test_clean = [clean_list[i] for i in test_idx]
    
    # Generate val and test manifests
    val_manifest_path = fold_dir / f"fold_{fold_idx}_val_manifest.jsonl"
    test_manifest_path = fold_dir / f"fold_{fold_idx}_test_manifest.jsonl"
    
    if not val_manifest_path.exists():
        print(f"  Fold {fold_idx}: Generating val manifest ({len(val_clean)} samples)...")
        generate_val_manifest(val_clean, noise_list, val_manifest_path, 
                             n_val=len(val_clean), seed=int(cfg.get("seed", 42)), 
                             segment_len=float(cfg.get("segment_len", 4.0)))
    else:
        print(f"  Fold {fold_idx}: Val manifest already exists")
    
    if not test_manifest_path.exists():
        print(f"  Fold {fold_idx}: Generating test manifest ({len(test_clean)} samples)...")
        generate_val_manifest(test_clean, noise_list, test_manifest_path, 
                             n_val=len(test_clean), seed=int(cfg.get("seed", 42)), 
                             segment_len=float(cfg.get("segment_len", 4.0)))
    else:
        print(f"  Fold {fold_idx}: Test manifest already exists")

print(f"\n[SUCCESS] All manifests ready for {n_splits} folds")


In [ ]:
from src.utils import list_audio_files, generate_val_manifest
from src.dataset import NoiseSuppressionDataset
from torch.utils.data import DataLoader
from sklearn.model_selection import KFold
import json
import torch

# Custom collate function to handle variable-length sequences
def collate_audio_batch(batch):
    """Pad variable-length audio samples to the same length"""
    max_clean_len = max(item['clean'].shape[-1] for item in batch)
    max_noisy_len = max(item['noisy'].shape[-1] for item in batch)
    max_len = max(max_clean_len, max_noisy_len)
    
    batch_clean = []
    batch_noisy = []
    batch_meta = []
    
    for item in batch:
        clean = item['clean']
        noisy = item['noisy']
        
        if clean.shape[-1] < max_len:
            pad_len = max_len - clean.shape[-1]
            clean = torch.nn.functional.pad(clean, (0, pad_len))
        if noisy.shape[-1] < max_len:
            pad_len = max_len - noisy.shape[-1]
            noisy = torch.nn.functional.pad(noisy, (0, pad_len))
        
        batch_clean.append(clean)
        batch_noisy.append(noisy)
        batch_meta.append(item.get('meta', ''))
    
    return {
        'clean': torch.stack(batch_clean),
        'noisy': torch.stack(batch_noisy),
        'meta': batch_meta
    }

clean_list = list_audio_files(clean_root)
noise_list = list_audio_files(noise_root)
print("#clean:", len(clean_list))
print("#noise:", len(noise_list))

assert len(clean_list) > 0 and len(noise_list) > 0

# K-Fold setup
n_splits = 5  # Number of folds
kfold = KFold(n_splits=n_splits, shuffle=True, random_state=int(cfg.get("seed", 42)))

# Create fold indices
clean_indices = list(range(len(clean_list)))
fold_dir = ROOT / "manifests"
fold_dir.mkdir(parents=True, exist_ok=True)

# Store fold indices for reproducibility
fold_indices = []
for fold_idx, (train_val_idx, test_idx) in enumerate(kfold.split(clean_indices)):
    fold_indices.append({
        "fold": fold_idx,
        "train_val": train_val_idx.tolist(),
        "test": test_idx.tolist(),
    })
    print(f"Fold {fold_idx}: train_val={len(train_val_idx)}, test={len(test_idx)}")

# Save fold indices
fold_indices_path = fold_dir / "fold_indices.json"
with open(fold_indices_path, "w") as f:
    json.dump(fold_indices, f, indent=2)
print(f"\nFold indices saved to {fold_indices_path}")

# Example: prepare dataloaders for fold 0
fold_idx = 0
fold_info = fold_indices[fold_idx]
train_val_idx = fold_info["train_val"]
test_idx = fold_info["test"]

# Split train_val into train and val (80/20 of remaining)
val_size = max(1, len(train_val_idx) // 5)
train_idx = train_val_idx[:-val_size]
val_idx = train_val_idx[-val_size:]

print(f"\nFold {fold_idx} split:")
print(f"  Train indices: {len(train_idx)}")
print(f"  Val indices: {len(val_idx)}")
print(f"  Test indices: {len(test_idx)}")

# Filter lists for this fold
train_clean = [clean_list[i] for i in train_idx]
val_clean = [clean_list[i] for i in val_idx]
test_clean = [clean_list[i] for i in test_idx]

mix_cfg = {
    "seed": int(cfg.get("seed", 42)),
    "p_rir": float(cfg.get("p_rir", 0.35)),
    "p_vary": float(cfg.get("p_vary", 0.4)),
    "project_root": str(ROOT),
}

# Generate val and test manifests for this fold
val_manifest_path = fold_dir / f"fold_{fold_idx}_val_manifest.jsonl"
test_manifest_path = fold_dir / f"fold_{fold_idx}_test_manifest.jsonl"

if not val_manifest_path.exists():
    generate_val_manifest(val_clean, noise_list, val_manifest_path, 
                         n_val=len(val_clean), seed=int(cfg.get("seed", 42)), 
                         segment_len=float(cfg.get("segment_len", 4.0)))

if not test_manifest_path.exists():
    generate_val_manifest(test_clean, noise_list, test_manifest_path, 
                         n_val=len(test_clean), seed=int(cfg.get("seed", 42)), 
                         segment_len=float(cfg.get("segment_len", 4.0)))

# Create datasets
train_ds = NoiseSuppressionDataset(train_clean, noise_list, 
                                   segment_len=float(cfg["segment_len"]), 
                                   sr=int(cfg["sr"]), mix_config=mix_cfg, mode="train")
val_ds = NoiseSuppressionDataset(val_clean, noise_list, 
                                 segment_len=float(cfg["segment_len"]), 
                                 sr=int(cfg["sr"]), mix_config=mix_cfg, 
                                 mode="val", manifest=val_manifest_path)
test_ds = NoiseSuppressionDataset(test_clean, noise_list, 
                                  segment_len=float(cfg["segment_len"]), 
                                  sr=int(cfg["sr"]), mix_config=mix_cfg, 
                                  mode="val", manifest=test_manifest_path)

# Create dataloaders with custom collate function (num_workers=0 for Windows stability)
train_loader = DataLoader(train_ds, batch_size=int(cfg["batch_size"]), shuffle=True, 
                         num_workers=0, pin_memory=False, drop_last=True, collate_fn=collate_audio_batch)
val_loader = DataLoader(val_ds, batch_size=int(cfg["batch_size"]), shuffle=False, 
                       num_workers=0, pin_memory=False, collate_fn=collate_audio_batch)
test_loader = DataLoader(test_ds, batch_size=int(cfg["batch_size"]), shuffle=False, 
                        num_workers=0, pin_memory=False, collate_fn=collate_audio_batch)

print(f"\nDataloaders created:")
print(f"  Train: {len(train_loader)} batches")
print(f"  Val: {len(val_loader)} batches")
print(f"  Test: {len(test_loader)} batches")

# Test a batch
batch = next(iter(train_loader))
print(f"\nBatch shapes: clean={batch['clean'].shape}, noisy={batch['noisy'].shape}")


## Auto-tuning

In [ ]:
## Hyperparameter Auto-Tuning Module

import torch
import torch.nn as nn
from torch.optim.lr_scheduler import CosineAnnealingLR, StepLR, LambdaLR
import numpy as np
from itertools import product
import json

class EarlyStopping:
    """Early stopping to prevent overfitting"""
    def __init__(self, patience=5, min_delta=0.0, restore_best_weights=True):
        self.patience = patience
        self.min_delta = min_delta
        self.restore_best_weights = restore_best_weights
        self.best_loss = None
        self.counter = 0
        self.best_weights = None
        
    def __call__(self, val_loss, model):
        if self.best_loss is None:
            self.best_loss = val_loss
            if self.restore_best_weights:
                self.best_weights = model.state_dict().copy()
        elif val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.counter = 0
            if self.restore_best_weights:
                self.best_weights = model.state_dict().copy()
        else:
            self.counter += 1
        
        return self.counter >= self.patience
    
    def restore(self, model):
        if self.best_weights is not None:
            model.load_state_dict(self.best_weights)

class LRSchedulerFactory:
    """Factory for creating different learning rate schedulers"""
    
    @staticmethod
    def create(scheduler_type, optimizer, **kwargs):
        """
        Create a learning rate scheduler
        
        Args:
            scheduler_type: 'cosine', 'step', 'triangular', 'exponential', or 'plateau'
            optimizer: PyTorch optimizer
            **kwargs: scheduler-specific parameters
        
        Returns:
            scheduler object or None
        """
        if scheduler_type == 'cosine':
            t_max = kwargs.get('t_max', 10)
            eta_min = kwargs.get('eta_min', 1e-5)
            return CosineAnnealingLR(optimizer, T_max=t_max, eta_min=eta_min)
        
        elif scheduler_type == 'step':
            step_size = kwargs.get('step_size', 3)
            gamma = kwargs.get('gamma', 0.1)
            return StepLR(optimizer, step_size=step_size, gamma=gamma)
        
        elif scheduler_type == 'exponential':
            gamma = kwargs.get('gamma', 0.95)
            return torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=gamma)
        
        elif scheduler_type == 'triangular':
            # Custom triangular scheduler
            base_lr = optimizer.defaults['lr']
            max_lr = kwargs.get('max_lr', base_lr * 10)
            cycle_size = kwargs.get('cycle_size', 10)
            
            def triangular_lr(epoch):
                cycle = epoch % cycle_size
                if cycle < cycle_size / 2:
                    return 1 + (max_lr / base_lr - 1) * (cycle / (cycle_size / 2))
                else:
                    return 1 + (max_lr / base_lr - 1) * (2 - cycle / (cycle_size / 2))
            
            return LambdaLR(optimizer, lr_lambda=triangular_lr)
        
        elif scheduler_type == 'plateau':
            patience = kwargs.get('patience', 10)
            factor = kwargs.get('factor', 0.1)
            return torch.optim.lr_scheduler.ReduceLROnPlateau(
                optimizer, mode='min', factor=factor, patience=patience, verbose=True
            )
        
        else:
            return None

class HyperparameterTuner:
    """Hyperparameter grid search and tuning"""
    
    def __init__(self, param_grid):
        """
        Initialize tuner with parameter grid
        
        Args:
            param_grid: dict of param_name -> list of values to try
            Example: {
                'lr': [1e-4, 3e-4, 1e-3],
                'batch_size': [16, 32, 64],
                'weight_decay': [0, 1e-5, 1e-4]
            }
        """
        self.param_grid = param_grid
        self.results = []
    
    def get_combinations(self):
        """Generate all parameter combinations"""
        keys = list(self.param_grid.keys())
        values = list(self.param_grid.values())
        
        for combination in product(*values):
            yield dict(zip(keys, combination))
    
    def get_best_params(self, metric='val_loss', top_k=1):
        """Get best parameters based on metric"""
        if not self.results:
            return None
        
        sorted_results = sorted(self.results, key=lambda x: x[metric])
        return [r['params'] for r in sorted_results[:top_k]]
    
    def save_results(self, filepath):
        """Save tuning results to JSON"""
        with open(filepath, 'w') as f:
            json.dump(self.results, f, indent=2)
    
    def load_results(self, filepath):
        """Load tuning results from JSON"""
        with open(filepath, 'r') as f:
            self.results = json.load(f)

# Example usage:
print("[INFO] Hyperparameter auto-tuning module loaded!")
print("\nAvailable schedulers: 'cosine', 'step', 'exponential', 'triangular', 'plateau'")
print("Example parameter grid:")
print({
    'lr': [1e-4, 3e-4, 1e-3],
    'scheduler_type': ['cosine', 'step'],
    'weight_decay': [0, 1e-5]
})


## Instantiate model + show parameter count

In [ ]:
import torch
from src.model import MobileDeepFilterNet, MobileDeepFilterNetConfig

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

m_cfg = MobileDeepFilterNetConfig(
    freq_bins=161,
    enc_channels=int(cfg.get("enc_channels", 32)),
    num_encoder_blocks=int(cfg.get("num_encoder_blocks", 2)),
    gru_hidden=int(cfg.get("gru_hidden", 64)),
    gru_layers=int(cfg.get("gru_layers", 1)),
    k_tap=int(cfg.get("k_tap", 3)),
)
model = MobileDeepFilterNet(m_cfg).to(device)

n_params = sum(p.numel() for p in model.parameters())
print("device:", device)
print("params:", n_params)
model

## K-Fold Training Loop (iterate through all folds)

Train and evaluate on each fold separately, storing metrics for each fold.


In [ ]:
## Metric Computation Utilities (PESQ, STOI)

import numpy as np
import warnings

def compute_pesq(reference, degraded, sr=16000):
    """Compute PESQ score (0-4.5, higher is better)"""
    try:
        from pesq import pesq as pesq_score
        # PESQ expects sample rate and audio signals
        score = pesq_score(sr, reference, degraded, mode='wb')
        return float(score)
    except ImportError:
        print("[WARNING] PESQ not installed, skipping PESQ metric")
        return None
    except Exception as e:
        print(f"[WARNING] PESQ computation failed: {e}")
        return None

def compute_stoi(reference, degraded, sr=16000):
    """Compute STOI score (0-1, higher is better)"""
    try:
        from pystoi import stoi
        # STOI expects audio signals and sample rate
        score = stoi(reference, degraded, sr)
        return float(score)
    except ImportError:
        print("[WARNING] STOI not installed, try: pip install pystoi")
        return None
    except Exception as e:
        print(f"[WARNING] STOI computation failed: {e}")
        return None

def compute_spectral_distortion(reference_spec, degraded_spec):
    """Compute spectral distortion in dB (works on log-power spectrograms)"""
    # Flatten spectrograms for comparison
    ref_flat = reference_spec.flatten()
    deg_flat = degraded_spec.flatten()
    
    # Mean squared error in dB scale
    mse = np.mean((ref_flat - deg_flat) ** 2)
    return float(mse)

def compute_snr_spectrogram(clean_spec, denoised_spec):
    """Compute SNR on spectrograms in dB scale"""
    # Assume clean_spec is target, denoised is estimate
    # SNR = power of clean / power of noise
    noise_spec = clean_spec - denoised_spec
    
    clean_power = np.mean(clean_spec ** 2)
    noise_power = np.mean(noise_spec ** 2)
    
    snr = 10 * np.log10(clean_power / (noise_power + 1e-8))
    return float(snr)

def compute_cosine_similarity(clean_spec, denoised_spec):
    """Compute cosine similarity between clean and denoised spectrograms (0-1, higher=better)"""
    # Flatten and normalize
    clean_flat = clean_spec.flatten()
    denoised_flat = denoised_spec.flatten()
    
    # Cosine similarity
    dot_product = np.dot(clean_flat, denoised_flat)
    norm_clean = np.linalg.norm(clean_flat)
    norm_denoised = np.linalg.norm(denoised_flat)
    
    similarity = dot_product / (norm_clean * norm_denoised + 1e-8)
    # Map from [-1, 1] to [0, 1] for easier interpretation
    similarity = (similarity + 1) / 2
    return float(similarity)

def compute_pearson_correlation(clean_spec, denoised_spec):
    """Compute Pearson correlation coefficient (0-1, higher=better)"""
    clean_flat = clean_spec.flatten()
    denoised_flat = denoised_spec.flatten()
    
    # Pearson correlation
    correlation = np.corrcoef(clean_flat, denoised_flat)[0, 1]
    # Handle NaN cases
    correlation = 0.0 if np.isnan(correlation) else correlation
    # Map from [-1, 1] to [0, 1]
    correlation = (correlation + 1) / 2
    return float(correlation)

def compute_mae_spectrogram(clean_spec, denoised_spec):
    """Compute mean absolute error between spectrograms (lower=better)"""
    mae = np.mean(np.abs(clean_spec - denoised_spec))
    return float(mae)

class MetricsTracker:
    """Track metrics across epochs and compute improvements"""
    def __init__(self):
        self.history = {
            'pesq': [],
            'stoi': [],
            'si_snr': [],
            'similarity': [],
            'correlation': [],
            'mae': [],
            'loss': []
        }
    
    def update(self, pesq=None, stoi=None, si_snr=None, similarity=None, correlation=None, mae=None, loss=None):
        """Record metrics for current epoch"""
        if pesq is not None:
            self.history['pesq'].append(pesq)
        if stoi is not None:
            self.history['stoi'].append(stoi)
        if si_snr is not None:
            self.history['si_snr'].append(si_snr)
        if similarity is not None:
            self.history['similarity'].append(similarity)
        if correlation is not None:
            self.history['correlation'].append(correlation)
        if mae is not None:
            self.history['mae'].append(mae)
        if loss is not None:
            self.history['loss'].append(loss)
    
    def get_improvement(self, metric_name, from_epoch=0):
        """Get percentage improvement from starting epoch to current"""
        values = self.history.get(metric_name, [])
        if len(values) <= from_epoch:
            return 0.0
        
        start_val = values[from_epoch]
        current_val = values[-1]
        
        if start_val == 0:
            return 0.0
        
        # For loss, lower is better, so negative improvement means reduction
        if metric_name == 'loss':
            improvement = ((start_val - current_val) / start_val) * 100
        # For PESQ/STOI/SI-SNR, higher is better
        else:
            improvement = ((current_val - start_val) / abs(start_val)) * 100
        
        return improvement

print("[INFO] Metrics utilities loaded!")
print("  - Spectral SNR (dB scale, higher=better)")
print("  - Spectral Distortion (lower=better)")
print("  - Cosine Similarity (0-1 scale, higher=better)")
print("  - Pearson Correlation (0-1 scale, higher=better)")
print("  - MAE (lower=better)")
print("  - Automatic improvement tracking")


In [ ]:
import json
import torch
import torch.nn as nn
from pathlib import Path
from src.model import MobileDeepFilterNet, MobileDeepFilterNetConfig

# Custom collate function
def collate_audio_batch(batch):
    """Pad variable-length audio samples to the same length"""
    max_clean_len = max(item['clean'].shape[-1] for item in batch)
    max_noisy_len = max(item['noisy'].shape[-1] for item in batch)
    max_len = max(max_clean_len, max_noisy_len)
    
    batch_clean = []
    batch_noisy = []
    batch_meta = []
    
    for item in batch:
        clean = item['clean']
        noisy = item['noisy']
        
        if clean.shape[-1] < max_len:
            pad_len = max_len - clean.shape[-1]
            clean = torch.nn.functional.pad(clean, (0, pad_len))
        if noisy.shape[-1] < max_len:
            pad_len = max_len - noisy.shape[-1]
            noisy = torch.nn.functional.pad(noisy, (0, pad_len))
        
        batch_clean.append(clean)
        batch_noisy.append(noisy)
        batch_meta.append(item.get('meta', ''))
    
    return {
        'clean': torch.stack(batch_clean),
        'noisy': torch.stack(batch_noisy),
        'meta': batch_meta
    }

# Feature extraction function
def waveform_to_features(audio_waveform, sr=16000, n_fft=512, hop_length=160, n_mels=161):
    """Convert audio waveform to log-power spectrogram."""
    if audio_waveform.dim() == 1:
        audio_waveform = audio_waveform.unsqueeze(0)
    
    stft = torch.stft(audio_waveform, n_fft=n_fft, hop_length=hop_length, 
                      return_complex=False, pad_mode='reflect')
    mag = torch.sqrt(stft[..., 0]**2 + stft[..., 1]**2 + 1e-8)
    mag = mag[:, :n_mels, :]
    feats_logp = 20 * torch.log10(mag.clamp(min=1e-5))
    return feats_logp.unsqueeze(1)

# Setup device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load fold indices
fold_indices_path = ROOT / "manifests" / "fold_indices.json"
with open(fold_indices_path, "r") as f:
    fold_indices = json.load(f)

n_splits = len(fold_indices)
fold_results = []

quick = False  # Set to True for quick test, False for full training
sr = int(cfg.get("sr", 16000))

print(f"\n{'='*60}")
print(f"K-Fold Cross-Validation Training (with Auto-Tuning)")
print(f"Folds: {n_splits} | Quick mode: {quick} | Device: {device}")
print(f"{'='*60}\n")

for fold_idx in range(n_splits):
    print(f"\n{'='*60}")
    print(f"Training Fold {fold_idx + 1}/{n_splits}")
    print(f"{'='*60}\n")
    
    fold_info = fold_indices[fold_idx]
    train_val_idx = fold_info["train_val"]
    test_idx = fold_info["test"]
    
    # Create train/val/test split for this fold
    val_size = max(1, len(train_val_idx) // 5)
    train_idx = train_val_idx[:-val_size]
    val_idx = train_val_idx[-val_size:]
    
    train_clean = [clean_list[i] for i in train_idx]
    val_clean = [clean_list[i] for i in val_idx]
    test_clean = [clean_list[i] for i in test_idx]
    
    print(f"Fold {fold_idx}: train={len(train_idx)}, val={len(val_idx)}, test={len(test_idx)}")
    
    # Prepare manifests
    fold_dir = ROOT / "manifests"
    val_manifest_path = fold_dir / f"fold_{fold_idx}_val_manifest.jsonl"
    test_manifest_path = fold_dir / f"fold_{fold_idx}_test_manifest.jsonl"
    
    # Create datasets
    train_ds = NoiseSuppressionDataset(train_clean, noise_list, 
                                       segment_len=float(cfg["segment_len"]), 
                                       sr=int(cfg["sr"]), mix_config=mix_cfg, mode="train")
    val_ds = NoiseSuppressionDataset(val_clean, noise_list, 
                                     segment_len=float(cfg["segment_len"]), 
                                     sr=int(cfg["sr"]), mix_config=mix_cfg, 
                                     mode="val", manifest=val_manifest_path)
    test_ds = NoiseSuppressionDataset(test_clean, noise_list, 
                                      segment_len=float(cfg["segment_len"]), 
                                      sr=int(cfg["sr"]), mix_config=mix_cfg, 
                                      mode="val", manifest=test_manifest_path)
    
    # Create dataloaders
    train_loader = DataLoader(train_ds, batch_size=int(cfg["batch_size"]), shuffle=True, 
                             num_workers=0, pin_memory=False, drop_last=True, collate_fn=collate_audio_batch)
    val_loader = DataLoader(val_ds, batch_size=int(cfg["batch_size"]), shuffle=False, 
                           num_workers=0, pin_memory=False, collate_fn=collate_audio_batch)
    test_loader = DataLoader(test_ds, batch_size=int(cfg["batch_size"]), shuffle=False, 
                            num_workers=0, pin_memory=False, collate_fn=collate_audio_batch)
    
    # Training config with auto-tuning
    epochs = 1 if quick else int(cfg.get("epochs", 10))
    lr = float(cfg.get("lr", 3e-4))
    weight_decay = float(cfg.get("weight_decay", 0.0))
    n_steps = min(10, len(train_loader)) if quick else len(train_loader)
    
    # Scheduler and stopping config
    scheduler_type = cfg.get("scheduler_type", "cosine")
    patience_early_stop = int(cfg.get("patience_early_stop", 5))
    
    print(f"Training config:")
    print(f"  Epochs: {epochs}")
    print(f"  Learning rate: {lr}")
    print(f"  Weight decay: {weight_decay}")
    print(f"  LR Scheduler: {scheduler_type}")
    print(f"  Early stopping patience: {patience_early_stop}")
    
    # Initialize model for this fold
    m_cfg = MobileDeepFilterNetConfig(
        freq_bins=161,
        enc_channels=int(cfg.get("enc_channels", 32)),
        num_encoder_blocks=int(cfg.get("num_encoder_blocks", 2)),
        gru_hidden=int(cfg.get("gru_hidden", 64)),
        gru_layers=int(cfg.get("gru_layers", 1)),
        k_tap=int(cfg.get("k_tap", 3)),
    )
    model = MobileDeepFilterNet(m_cfg).to(device)
    
    # Setup optimizer with weight decay
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    criterion = nn.MSELoss()
    
    # Create learning rate scheduler
    scheduler = LRSchedulerFactory.create(scheduler_type, optimizer, 
                                          t_max=epochs, 
                                          step_size=max(1, epochs//3),
                                          cycle_size=epochs)
    
    # Initialize early stopping
    early_stopper = EarlyStopping(patience=patience_early_stop, min_delta=1e-4, restore_best_weights=True)
    
    # Initialize metrics tracker
    metrics_tracker = MetricsTracker()
    
    # Training loop with auto-tuning features
    best_val_loss = float('inf')
    all_train_loss = []
    all_val_loss = []
    
    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        batch_count = 0
        
        for batch_idx, batch in enumerate(train_loader):
            if batch_idx >= n_steps:
                break
            
            clean = batch['clean'].to(device)
            noisy = batch['noisy'].to(device)
            feats_noisy = waveform_to_features(noisy, sr=sr).to(device)
            feats_clean = waveform_to_features(clean, sr=sr).to(device)
            
            optimizer.zero_grad()
            mask, w_taps, _ = model(feats_noisy)
            denoised = feats_noisy.squeeze(1) * mask
            loss = criterion(denoised, feats_clean.squeeze(1))
            
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # Gradient clipping
            optimizer.step()
            
            train_loss += loss.item()
            batch_count += 1
            
            if (batch_idx + 1) % max(1, max(1, n_steps // 3)) == 0:
                print(f"  Epoch {epoch+1}/{epochs} - Step {batch_idx+1}/{n_steps} - Loss: {loss.item():.4f}")
        
        avg_train_loss = train_loss / batch_count if batch_count > 0 else 0
        all_train_loss.append(avg_train_loss)
        
        # Validation phase with metrics computation
        model.eval()
        val_loss = 0.0
        val_batch_count = 0
        
        # Metrics computation on subset (first 3 batches for speed)
        pesq_scores = []
        stoi_scores = []
        si_snr_scores = []
        similarity_scores = []
        correlation_scores = []
        mae_scores = []
        
        with torch.no_grad():
            for batch_idx, batch in enumerate(val_loader):
                clean = batch['clean'].to(device)
                noisy = batch['noisy'].to(device)
                feats_noisy = waveform_to_features(noisy, sr=sr).to(device)
                feats_clean = waveform_to_features(clean, sr=sr).to(device)
                mask, w_taps, _ = model(feats_noisy)
                denoised = feats_noisy.squeeze(1) * mask
                loss = criterion(denoised, feats_clean.squeeze(1))
                val_loss += loss.item()
                val_batch_count += 1
                
                # Compute metrics on first few batches (avoid excessive computation)
                if batch_idx < 3:
                    # Compute metrics on spectrograms (not waveforms)
                    for i in range(min(1, feats_clean.shape[0])):  # First sample in batch
                        clean_spec = feats_clean[i].squeeze(0).cpu().numpy()  # Remove batch dim
                        denoised_spec = denoised[i].cpu().numpy()
                        
                        # Compute spectral SNR (always available, works on spectrograms)
                        spec_snr = compute_snr_spectrogram(clean_spec, denoised_spec)
                        si_snr_scores.append(spec_snr)
                        
                        # Compute spectral distortion
                        spec_dist = compute_spectral_distortion(clean_spec, denoised_spec)
                        pesq_scores.append(spec_dist)
                        
                        # Compute similarity metrics (how similar denoised is to clean)
                        similarity = compute_cosine_similarity(clean_spec, denoised_spec)
                        similarity_scores.append(similarity)
                        
                        correlation = compute_pearson_correlation(clean_spec, denoised_spec)
                        correlation_scores.append(correlation)
                        
                        mae = compute_mae_spectrogram(clean_spec, denoised_spec)
                        mae_scores.append(mae)
        
        avg_val_loss = val_loss / val_batch_count if val_batch_count > 0 else 0
        all_val_loss.append(avg_val_loss)
        
        # Average metrics
        avg_pesq = np.mean(pesq_scores) if pesq_scores else None
        avg_stoi = np.mean(stoi_scores) if stoi_scores else None
        avg_si_snr = np.mean(si_snr_scores) if si_snr_scores else None
        avg_similarity = np.mean(similarity_scores) if similarity_scores else None
        avg_correlation = np.mean(correlation_scores) if correlation_scores else None
        avg_mae = np.mean(mae_scores) if mae_scores else None
        
        # Update metrics tracker
        metrics_tracker.update(pesq=avg_pesq, stoi=avg_stoi, si_snr=avg_si_snr, 
                              similarity=avg_similarity, correlation=avg_correlation, 
                              mae=avg_mae, loss=avg_val_loss)
        
        # Update learning rate
        if scheduler is not None:
            if isinstance(scheduler, torch.optim.lr_scheduler.ReduceLROnPlateau):
                scheduler.step(avg_val_loss)
            else:
                scheduler.step()
        
        # Log current learning rate
        current_lr = optimizer.param_groups[0]['lr']
        
        # Build metrics log string
        metrics_log = f"  Epoch {epoch+1} - Loss: {avg_val_loss:.4f}, LR: {current_lr:.2e}"
        
        # Add quality metrics and improvements
        if avg_pesq is not None:
            spec_dist_improvement = metrics_tracker.get_improvement('pesq', from_epoch=0) if epoch > 0 else 0
            metrics_log += f" | Spec.Dist: {avg_pesq:.2f}"
            if spec_dist_improvement != 0:
                metrics_log += f" ({spec_dist_improvement:+.1f}%)"
        
        if avg_si_snr is not None:
            spec_snr_improvement = metrics_tracker.get_improvement('si_snr', from_epoch=0) if epoch > 0 else 0
            metrics_log += f" | Spec.SNR: {avg_si_snr:.2f}dB"
            if spec_snr_improvement != 0:
                metrics_log += f" ({spec_snr_improvement:+.1f}%)"
        
        # Add similarity metrics
        if avg_similarity is not None:
            similarity_improvement = metrics_tracker.get_improvement('similarity', from_epoch=0) if epoch > 0 else 0
            metrics_log += f" | Similarity: {avg_similarity:.3f}"
            if similarity_improvement != 0:
                metrics_log += f" ({similarity_improvement:+.1f}%)"
        
        if avg_correlation is not None:
            correlation_improvement = metrics_tracker.get_improvement('correlation', from_epoch=0) if epoch > 0 else 0
            metrics_log += f" | Correlation: {avg_correlation:.3f}"
            if correlation_improvement != 0:
                metrics_log += f" ({correlation_improvement:+.1f}%)"
        
        if avg_mae is not None:
            mae_improvement = metrics_tracker.get_improvement('mae', from_epoch=0) if epoch > 0 else 0
            metrics_log += f" | MAE: {avg_mae:.3f}"
            if mae_improvement != 0:
                metrics_log += f" ({mae_improvement:+.1f}%)"
        
        # Loss improvement
        loss_improvement = metrics_tracker.get_improvement('loss', from_epoch=0) if epoch > 0 else 0
        if loss_improvement != 0:
            metrics_log += f" [Loss ↓ {loss_improvement:.1f}%]"
        
        print(metrics_log)
        
        # Check early stopping
        if early_stopper(avg_val_loss, model):
            print(f"  Early stopping triggered at epoch {epoch+1}")
            early_stopper.restore(model)
            break
        
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
    
    # Save fold checkpoint
    fold_ckpt_dir = ROOT / "checkpoints" / f"fold_{fold_idx}"
    fold_ckpt_dir.mkdir(parents=True, exist_ok=True)
    ckpt_path = fold_ckpt_dir / "best.pth"
    
    torch.save({
        "model_cfg": {
            "freq_bins": m_cfg.freq_bins,
            "enc_channels": m_cfg.enc_channels,
            "num_encoder_blocks": m_cfg.num_encoder_blocks,
            "gru_hidden": m_cfg.gru_hidden,
            "gru_layers": m_cfg.gru_layers,
            "k_tap": m_cfg.k_tap,
        },
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "best_val_loss": best_val_loss,
    }, str(ckpt_path))
    
    print(f"\n✓ Fold {fold_idx} training complete!")
    print(f"  Best val loss: {best_val_loss:.4f}")
    print(f"  Checkpoint saved: {ckpt_path}\n")
    
    # Save fold result with training metrics
    fold_results.append({
        "fold": fold_idx,
        "train_size": len(train_idx),
        "val_size": len(val_idx),
        "test_size": len(test_idx),
        "checkpoint": str(ckpt_path),
        "best_val_loss": float(best_val_loss),
        "final_train_loss": float(avg_train_loss),
        "learning_rate": lr,
        "weight_decay": weight_decay,
        "scheduler": scheduler_type,
        "status": "completed"
    })
    
    # Clean up
    del model
    torch.cuda.empty_cache()

# Summary
print(f"\n{'='*60}")
print(f"K-Fold Training Summary ({n_splits} folds)")
print(f"{'='*60}\n")

for result in fold_results:
    print(f"Fold {result['fold']}: val_loss={result['best_val_loss']:.4f} | "
          f"train={result['train_size']}, val={result['val_size']}, test={result['test_size']}")

# Save results with auto-tuning metrics
results_path = ROOT / "manifests" / "kfold_results.json"
with open(results_path, "w") as f:
    json.dump(fold_results, f, indent=2)
print(f"\nResults saved to {results_path}")

# Find best hyperparameters across folds
best_fold = min(fold_results, key=lambda x: x['best_val_loss'])
print(f"\nBest fold: Fold {best_fold['fold']} with val_loss={best_fold['best_val_loss']:.4f}")
print(f"  Learning rate: {best_fold['learning_rate']}")
print(f"  Weight decay: {best_fold['weight_decay']}")
print(f"  Scheduler: {best_fold['scheduler']}")

print(f"\n{'='*60}")
print(f"All {n_splits} folds completed! ✓")
print(f"{'='*60}")

## Run unit tests (mixing/SNR correctness)

In [ ]:
!python -m unittest discover -s "E:\\0. Noise reduce\\tests" -p "test_*.py" -v

## Export best model (TorchScript + ONNX)

This exports a wrapper that takes log-power features and outputs (mask, taps).

In [ ]:
import torch
from src.model import MobileDeepFilterNet, MobileDeepFilterNetConfig

ckpt = ROOT / "checkpoints" / "best.pth"

# If checkpoint doesn't exist, create one from the current model (cell 13)
if not ckpt.exists():
    print(f"[INFO] Checkpoint not found at {ckpt}")
    print("[INFO] Creating checkpoint from current model (cell 13)...")
    
    ckpt.parent.mkdir(parents=True, exist_ok=True)
    
    # Save current model from cell 13
    checkpoint = {
        "model_cfg": {
            "freq_bins": m_cfg.freq_bins,
            "enc_channels": m_cfg.enc_channels,
            "num_encoder_blocks": m_cfg.num_encoder_blocks,
            "gru_hidden": m_cfg.gru_hidden,
            "gru_layers": m_cfg.gru_layers,
            "k_tap": m_cfg.k_tap,
        },
        "model": model.cpu().state_dict(),
    }
    torch.save(checkpoint, str(ckpt))
    print(f"[SUCCESS] Checkpoint saved to {ckpt}")
    # Move model back to device
    model = model.to(device)
else:
    print(f"[INFO] Using existing checkpoint: {ckpt}")

# Load checkpoint
state = torch.load(str(ckpt), map_location="cpu")
model_cfg = MobileDeepFilterNetConfig(**state["model_cfg"])
m = MobileDeepFilterNet(model_cfg).eval()
m.load_state_dict(state["model"], strict=True)

class ExportWrapper(torch.nn.Module):
    def __init__(self, model: torch.nn.Module):
        super().__init__()
        self.model = model
    def forward(self, feats_logp: torch.Tensor):
        mask, w_taps, _ = self.model(feats_logp, None)
        return mask, w_taps

wrapper = ExportWrapper(m).eval()
example = torch.randn(1, 1, 161, 64)

ts_path = ROOT / "checkpoints" / "mobiledeepfilternet_best.ts"
torch.jit.trace(wrapper, (example,)).save(str(ts_path))
print("Saved:", ts_path)

onnx_path = ROOT / "checkpoints" / "mobiledeepfilternet_best.onnx"
torch.onnx.export(
    wrapper,
    (example,),
    str(onnx_path),
    input_names=["feats_logp"],
    output_names=["mask", "w_taps"],
    opset_version=17,
    dynamic_axes={"feats_logp": {0: "B", 3: "T"}, "mask": {0: "B", 2: "T"}, "w_taps": {0: "B", 3: "T"}},
)
print("Saved:", onnx_path)